# Preprocessing

In [1]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

In [2]:
#Storing the movie information into a pandas dataframe
movies_df = pd.read_csv("Desktop\\movies.csv")
#Storing the user information into a pandas dataframe
ratings_df = pd.read_csv("Desktop\\ratings.csv")
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


#### Removing the year from the "title" column and storing in a new "year" column

In [3]:
#We specify the parantheses so we don't conflict with movies that have years in their titles(regex style)
movies_df["year"] = movies_df.title.str.extract("(\(\d\d\d\d\))",expand = False)
#Removing the parantheses
movies_df["year"] = movies_df.year.str.extract("(\d\d\d\d)",expand = False)
#Removing the years from the "title" column
movies_df["title"] = movies_df.title.str.replace("(\(\d\d\d\d\))", "", regex = True)
#Applying the strip function to get rid of any ending whitespace characters
movies_df["title"] = movies_df["title"].apply(lambda x: x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


#### Dropping the genres column since we don't need it for this particular recommendation system

In [4]:
movies_df = movies_df.drop(columns = ["genres"])
movies_df.head()

,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


 #### ratings dataframe

In [5]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
#We don't need timestamp column
ratings_df = ratings_df.drop(columns = ["timestamp"])
ratings_df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


# Collaborative Filtering

In [7]:
user_input = [
    {'title':'Breakfast Club', 'rating':5},
    {'title':'Toy Story', 'rating':3.5},
    {'title':'Jumanji', 'rating':2},
    {'title':'Pulp Fiction', 'rating':5},
    {'title':'Akira', 'rating':4.5}
]
input_movies = pd.DataFrame(user_input)
input_movies

,title,rating
0,Breakfast Club,5.0
1,Toy Story,3.5
2,Jumanji,2.0
3,Pulp Fiction,5.0
4,Akira,4.5


#### Add movieId to input user

In [8]:
#Filtering out the movies by title
input_id = movies_df[movies_df["title"].isin(input_movies["title"].tolist())]
#Then merging it so we can get the movieID
input_movies = pd.merge(input_id, input_movies)
#Dropping information we won't use
input_movies = input_movies.drop(columns = ["year"])
input_movies

,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,296,Pulp Fiction,5.0
3,1274,Akira,4.5


#### Searching for the users has seen the same movies

In [9]:
user_subset = ratings_df[ratings_df["movieId"].isin(input_movies["movieId"].tolist())]
user_subset.head()

,userId,movieId,rating
0,1,1,4.0
16,1,296,3.0
320,4,296,1.0
516,5,1,4.0
533,5,296,5.0


#### Grouping up the rows by user ID

In [10]:
user_subset_group = user_subset.groupby(["userId"])

In [11]:
user_subset_group.get_group(5)

,userId,movieId,rating
516,5,1,4.0
533,5,296,5.0


In [12]:
user_subset_group = sorted(user_subset_group, key = lambda x: len(x[1]), reverse = True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_12012\2700256539.py:1: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  user_subset_group = sorted(user_subset_group, key = lambda x: len(x[1]), reverse = True)


#### Selecting a subset of users to iterate through.This limit is imposed because we don't want to waste too much going through every single user

In [13]:
user_subset_group = user_subset_group[0:100]

#### Calculating the Pearson Correlation between input user and subset group and store it in a dictionary where the key is the user Id and the value is the coefficient

In [14]:
pearson_correlation_dict = {}

#For every user group in our subset
for name, group in user_subset_group:
    #Sorting the input and current user group so the values aren't mixed up
    group = group.sort_values(by='movieId')
    input_movies = input_movies.sort_values(by='movieId')
    #Getting the n for the formula
    n_ratings = len(group)
    #Getting the review scores for the movies that they both have in common
    temp_df = input_movies[input_movies['movieId'].isin(group['movieId'].tolist())]
    temp_rating_list = temp_df['rating'].tolist()
    temp_group_list = group['rating'].tolist()
    #3 factors of pearson
    sxx = sum([i**2 for i in temp_rating_list]) - pow(sum(temp_rating_list),2)/float(n_ratings)
    syy = sum([i**2 for i in  temp_group_list]) - pow(sum( temp_group_list),2)/float(n_ratings)
    sxy = sum(i*j for i, j in zip(temp_rating_list, temp_group_list)) - sum(temp_rating_list)*sum(temp_group_list)/float(n_ratings)
    if sxx != 0 and syy!= 0:
        pearson_correlation_dict[name] = sxy/sqrt(sxx*syy)
    else:
        pearson_correlation_dict[name] = 0

In [15]:
pearson_correlation_dict

{91: 0.9221388919541469,
 177: 0.0657951694959769,
 219: 0.5459486832355505,
 274: 0.8510644963469901,
 298: 0.9883173560569456,
 414: 0.9258200997725514,
 434: 0.9864036607532465,
 474: 0.0657951694959769,
 477: 0.7237468644557459,
 480: 0.8728715609439696,
 483: 0.35043832202523123,
 599: 0.9341484842923421,
 600: 0.18442777839082938,
 608: 0.9378934722869389,
 18: 1.0,
 21: 0,
 50: 0.9449111825230734,
 57: -0.9449111825230682,
 68: -0.8660254037844356,
 103: 0.8660254037844402,
 107: -1.0,
 135: 0.8660254037844402,
 140: 0.5,
 144: 1.0,
 153: 0.8660254037844379,
 160: 0.8660254037844402,
 182: 0.9449111825230684,
 202: 0,
 217: 0.0,
 226: 0.9819805060619667,
 232: 0.6546536707079778,
 240: -0.8660254037844386,
 249: 0,
 288: 0.9332565252573829,
 304: 0.8660254037844356,
 307: 0.9607689228305233,
 318: 0.88249750329277,
 322: 0.9607689228305233,
 323: 0.0,
 330: 0.8660254037844386,
 353: 0.8660254037844356,
 357: 0.7205766921228925,
 359: 0.8660254037844448,
 372: 0.142857142857144,


In [16]:
pearson_df = pd.DataFrame.from_dict(pearson_correlation_dict, orient='index')
pearson_df.columns = ['similarity_index']
pearson_df['userId'] = pearson_df.index
pearson_df.index = range(len(pearson_df))
pearson_df.head()

,similarity_index,userId
0,0.922139,91
1,0.065795,177
2,0.545949,219
3,0.851064,274
4,0.988317,298


In [17]:
top_users = pearson_df.sort_values(by = "similarity_index", ascending=False)[0:50]
top_users.head()

,similarity_index,userId
72,1.0,33
14,1.0,18
88,1.0,112
90,1.0,119
87,1.0,105


In [18]:
top_users_rating = top_users.merge(ratings_df, left_on='userId', right_on='userId', how='inner')
top_users_rating.head()

,similarity_index,userId,movieId,rating
0,1.0,33,1,3.0
1,1.0,33,7,1.0
2,1.0,33,11,2.0
3,1.0,33,17,4.0
4,1.0,33,21,4.0


In [19]:
#Multiplies the similarity by the user's ratings
top_users_rating["weighted_rating"] = top_users_rating["similarity_index"] * top_users_rating["rating"]
top_users_rating.head()

,similarity_index,userId,movieId,rating,weighted_rating
0,1.0,33,1,3.0,3.0
1,1.0,33,7,1.0,1.0
2,1.0,33,11,2.0,2.0
3,1.0,33,17,4.0,4.0
4,1.0,33,21,4.0,4.0


In [20]:
#Applying a sum to the top_users after grouping itup by userId
temp_top_users_rating = top_users_rating.groupby("movieId").sum()[["similarity_index","weighted_rating"]]
temp_top_users_rating.columns = ["sum_similarity_index","sum_weighted_rating"]
temp_top_users_rating.head()

,sum_similarity_index,sum_weighted_rating
movieId,,
1,41.894371,144.593560
2,28.907889,84.133978
3,14.161698,43.039994
5,9.283947,26.898021
6,18.893662,74.664086


In [21]:
#Creating an empty dataframe
recommendation_df = pd.DataFrame()
#Taking the weighted average
recommendation_df["weighted average recommendation score"] = temp_top_users_rating["sum_weighted_rating"] / temp_top_users_rating["sum_similarity_index"]
recommendation_df["movieId"] = temp_top_users_rating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,3.451384,1
2,2.910416,2
3,3.039183,3
5,2.897261,5
6,3.951806,6


In [22]:
recommendation_df = recommendation_df.sort_values(by = "weighted average recommendation score")
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
44243,0.5,44243
5356,0.5,5356
3774,0.5,3774
74275,0.5,74275
7636,0.5,7636
6557,0.5,6557
145724,0.5,145724
1453,0.5,1453
5323,0.5,5323


In [23]:
movies_df.loc[movies_df["movieId"].isin(recommendation_df.head(10)["movieId"].tolist())]

,movieId,title,year
1114,1453,"Beautician and the Beast, The",1997
2825,3774,House Party 2,1991
3804,5323,Jason X,2002
3821,5356,"Giant Spider Invasion, The",1975
4439,6557,Born to Be Wild,1995
4980,7636,Raising Cain,1992
6160,44243,Leprechaun 4: In Space,1997
7223,73319,Leap Year,2010
7251,74275,I Love You Phillip Morris,2009
9117,145724,Idaho Transfer,1973
